In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [ ]:
# Cargar variables de entorno desde el archivo .env
load_dotenv()

# Stage
cadena_stage = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS')};"
    f"UID={os.getenv('SQLSERVER_USUARIO')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    f"TrustServerCertificate=yes;"
)

engine_stage = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_stage}"
)


# DW

cadena_dw = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER_DW')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR_DW')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS_DW')};"
    f"UID={os.getenv('SQLSERVER_USUARIO_DW')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD_DW')};"
    f"TrustServerCertificate=yes;"
)

engine_dw = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_dw}"
)

In [ ]:
import os
import pandas as pd


# ============================================================
# 1. LEER VENTAS DESDE STAGE
# ============================================================

esquema_stage = os.getenv("SQLSERVER_ESQUEMA")

consulta_venta = f"""
SELECT
    venta_id,
    cliente_id,
    fecha_venta
FROM {esquema_stage}.Venta
WHERE estado = 'COMPLETADA'
"""

df_venta = pd.read_sql(
    consulta_venta,
    engine_stage
)


# ============================================================
# 2. LEER DETALLE DE VENTAS DESDE STAGE
# ============================================================

consulta_detalle = f"""
SELECT
    detalle_venta_id,
    venta_id,
    producto_id,
    cantidad,
    precio_unitario,
    costo_unitario,
    descuento,
    subtotal
FROM {esquema_stage}.DetalleVenta
"""

df_detalle = pd.read_sql(
    consulta_detalle,
    engine_stage
)


# ============================================================
# 3. UNIR CABECERA Y DETALLE
# ============================================================

df_fact = pd.merge(
    df_detalle,
    df_venta,
    on="venta_id",
    how="inner"
)


# ============================================================
# 4. OBTENER CLAVES DE DIMCLIENTE
# ============================================================

esquema_dw = os.getenv("SQLSERVER_ESQUEMA_DW")

df_dim_cliente = pd.read_sql(
    f"""
    SELECT
        cliente_sk,
        cliente_id_origen
    FROM {esquema_dw}.DimCliente
    """,
    engine_dw
)

df_fact = pd.merge(
    df_fact,
    df_dim_cliente,
    left_on="cliente_id",
    right_on="cliente_id_origen",
    how="left"
)


# ============================================================
# 5. OBTENER CLAVES DE DIMPRODUCTO
# ============================================================

df_dim_producto = pd.read_sql(
    f"""
    SELECT
        producto_sk,
        producto_id_origen
    FROM {esquema_dw}.DimProducto
    """,
    engine_dw
)

df_fact = pd.merge(
    df_fact,
    df_dim_producto,
    left_on="producto_id",
    right_on="producto_id_origen",
    how="left"
)


# ============================================================
# 6. GENERAR LA CLAVE DE FECHA
# ============================================================

df_fact["fecha_venta"] = pd.to_datetime(
    df_fact["fecha_venta"]
)

df_fact["fecha_sk"] = (
    df_fact["fecha_venta"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)


# ============================================================
# 7. VALIDAR QUE EXISTAN LAS CLAVES DE LAS DIMENSIONES
# ============================================================

df_fact = df_fact.dropna(
    subset=[
        "cliente_sk",
        "producto_sk",
        "fecha_sk"
    ]
)

df_fact["cliente_sk"] = df_fact["cliente_sk"].astype(int)
df_fact["producto_sk"] = df_fact["producto_sk"].astype(int)


# ============================================================
# 8. CALCULAR LAS MEDIDAS
# ============================================================

df_fact["importe_venta"] = df_fact["subtotal"]

df_fact["costo_total"] = (
    df_fact["cantidad"]
    * df_fact["costo_unitario"]
)

df_fact["utilidad"] = (
    df_fact["importe_venta"]
    - df_fact["costo_total"]
)


# ============================================================
# 9. PREPARAR LAS COLUMNAS DE FACTVENTAS
# ============================================================

df_fact = df_fact.rename(
    columns={
        "venta_id": "venta_id_origen",
        "detalle_venta_id": "detalle_venta_id_origen"
    }
)

df_fact = df_fact[
    [
        "venta_id_origen",
        "detalle_venta_id_origen",
        "cliente_sk",
        "producto_sk",
        "fecha_sk",
        "cantidad",
        "precio_unitario",
        "costo_unitario",
        "descuento",
        "importe_venta",
        "costo_total",
        "utilidad"
    ]
].copy()


# ============================================================
# 10. OBTENER LOS DETALLES YA CARGADOS EN FACTVENTAS
# ============================================================

df_fact_existente = pd.read_sql(
    f"""
    SELECT detalle_venta_id_origen
    FROM {esquema_dw}.FactVentas
    """,
    engine_dw
)


# ============================================================
# 11. IDENTIFICAR SOLO VENTAS NUEVAS
# ============================================================

df_fact_nueva = df_fact[
    ~df_fact["detalle_venta_id_origen"].isin(
        df_fact_existente["detalle_venta_id_origen"]
    )
].copy()

print(
    "Filas nuevas para FactVentas:",
    len(df_fact_nueva)
)


# ============================================================
# 12. CARGA INCREMENTAL
# ============================================================

if not df_fact_nueva.empty:

    df_fact_nueva.to_sql(
        name="FactVentas",
        con=engine_dw,
        schema=esquema_dw,
        if_exists="append",
        index=False
    )

    print(
        f"{len(df_fact_nueva)} filas cargadas en FactVentas."
    )

else:
    print("No existen ventas nuevas para cargar.")


# ============================================================
# 13. VERIFICAR RESULTADO
# ============================================================

df_resultado = pd.read_sql(
    f"""
    SELECT TOP 20 *
    FROM {esquema_dw}.FactVentas
    ORDER BY venta_sk
    """,
    engine_dw
)

display(df_resultado)